In [ ]:
# --- DEPENDECIES --- #

import os
import sys
import time
import copy
import random
import zipfile
from PIL import Image
import logging
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from dataclasses import dataclass
from urllib.request import urlretrieve
from tqdm import tqdm
from typing import Optional, Dict
import torch
import torch.nn as nn
from torchvision import datasets, models, transforms

In [ ]:
# --- LOGGING INFRASTRUCTURE --- #
def setup_logger(name: str = "Factory_CV"):
    """
    Configures a logger that outputs to Console.
    """
    logger = logging.getLogger(name)
    if logger.hasHandlers():
        logger.handlers.clear()

    logger.setLevel(logging.INFO)
    logger.propagate = False #Stopping Google Colab hidden logger

    # Format: Time - Level - Message
    formatter = logging.Formatter('%(asctime)s - [%(levelname)s] - %(message)s', datefmt='%H:%M:%S')

    ch = logging.StreamHandler(sys.stdout)
    ch.setFormatter(formatter)
    logger.addHandler(ch)
    return logger

logger = setup_logger()

In [ ]:
# --- CONFIGURATION  --- #
@dataclass
class ExperimentConfig:

    # Data
    DATA_URL: str = "https://download.pytorch.org/tutorial/hymenoptera_data.zip"
    DATA_DIR: Path = Path("hymenoptera_data")
    BATCH_SIZE: int = 32
    NUM_WORKERS: int = 2

    # Model
    NUM_CLASSES: int = 2
    FINE_TUNE: bool = False # False = Freeze feature extractor

    # Training
    EPOCHS: int = 25
    LEARNING_RATE: float = 0.001
    STEP_SIZE: int = 7
    GAMMA: float = 0.1
    SEED: int = 42

In [ ]:
# --- THE CLASS (The Machine) --- #
class BeeAntClassifier:
    def __init__(self, config: ExperimentConfig):
        """
        Initializes the Classifier.
        Sets up hardware, reproducibility, and logging.
        """
        self.cfg = config
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

        self._set_seed()
        logger.info(f"System initialized on: {self.device}")

        # Placeholders for state
        self.model = None
        self.dataloaders = {}
        self.dataset_sizes = {}
        self.class_names = []

    def _set_seed(self):
        """Ensures reproducibility."""
        seed = self.cfg.SEED
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

    def prepare_data(self):
        """
        Downloads data (if needed) and creates DataLoaders.
        """
        # Download & Extract
        if not self.cfg.DATA_DIR.exists():

            logger.info("Downloading dataset...")
            zip_path = "data_temp.zip"
            urlretrieve(self.cfg.DATA_URL, zip_path)
            with zipfile.ZipFile(zip_path, 'r') as zip_ref:
                zip_ref.extractall(".")
            os.remove(zip_path)

            logger.info("Dataset extracted.")
        else:
            logger.info("Dataset found. Skipping download.")

        # Define Transforms
        # We define separate pipelines for training and validation to prevent data leakage
        # and improve model generalization via Data Augmentation.
        data_transforms = {
            'train': transforms.Compose([
                # Data Augmentation: Artificially increase dataset diversity
                transforms.RandomResizedCrop(224),     # Solves aspect ratio issues & adds scale invariance
                transforms.RandomHorizontalFlip(),     # Adds symmetry invariance
                transforms.RandomRotation(15),         # Adds rotation invariance (+/- 15 degrees)
                transforms.ColorJitter(brightness=0.1, contrast=0.1), # Robustness to lighting conditions
                transforms.ToTensor(),
                # Normalization: Statistical standardization using ImageNet mean/std
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ]),
            'val': transforms.Compose([
                # Validation Pipeline: Deterministic transformations only
                transforms.Resize(256),                # Resize shorter side to 256px
                transforms.CenterCrop(224),            # Crop center square (standard ImageNet protocol)
                transforms.ToTensor(),
                transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
            ]),
        }

        # Create Datasets & Loaders
        image_datasets = {
            x: datasets.ImageFolder(str(self.cfg.DATA_DIR / x), data_transforms[x])
            for x in ['train', 'val']
        }

        self.dataloaders = {
            x: torch.utils.data.DataLoader(
                image_datasets[x],
                batch_size=self.cfg.BATCH_SIZE,
                shuffle=(x == 'train'),
                num_workers=self.cfg.NUM_WORKERS
            )
            for x in ['train', 'val']
        }

        self.dataset_sizes = {x: len(image_datasets[x]) for x in ['train', 'val']}
        self.class_names = image_datasets['train'].classes

        logger.info(f"Data Loaded. Classes: {self.class_names}")

    def build_model(self):
        """
        Loads ResNet18 and modifies the head for our specific task.
        """
        logger.info("Building Model Architecture...")
        weights = models.ResNet18_Weights.IMAGENET1K_V1
        self.model = models.resnet18(weights=weights)

        # Freeze layers if not fine-tuning
        if not self.cfg.FINE_TUNE:
            for param in self.model.parameters():
                param.requires_grad = False

        # Replace Head
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, self.cfg.NUM_CLASSES)

        self.model = self.model.to(self.device)

        # Setup Optimization Tools
        self.criterion = nn.CrossEntropyLoss()

        # Optimize only parameters that require gradients
        params_to_update = [p for p in self.model.parameters() if p.requires_grad]
        self.optimizer = torch.optim.Adam(params_to_update, lr=self.cfg.LEARNING_RATE)
        self.scheduler = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=self.cfg.STEP_SIZE, gamma=self.cfg.GAMMA
        )
        logger.info("Model Ready.")

    def train(self):
        """
        Main Training Loop with Checkpointing.
        """
        if self.model is None:
            raise RuntimeError("Model not built. Call build_model() first.")

        since = time.time()
        best_model_wts = copy.deepcopy(self.model.state_dict())
        best_acc = 0.0

        logger.info(f"Starting Training for {self.cfg.EPOCHS} epochs...")

        for epoch in range(self.cfg.EPOCHS):
            logger.info(f"--- Epoch {epoch + 1}/{self.cfg.EPOCHS} ---")

            for phase in ['train', 'val']:
                if phase == 'train':
                    self.model.train()
                else:
                    self.model.eval()

                running_loss = 0.0
                running_corrects = 0

                # TQDM for progress bar
                pbar = tqdm(self.dataloaders[phase], leave=False, desc=f"{phase}")

                for inputs, labels in pbar:
                    inputs = inputs.to(self.device)
                    labels = labels.to(self.device)

                    self.optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = self.model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = self.criterion(outputs, labels)

                        if phase == 'train':
                            loss.backward()
                            self.optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)

                    # Update progress bar
                    pbar.set_postfix(loss=loss.item())

                if phase == 'train':
                    self.scheduler.step()

                epoch_loss = running_loss / self.dataset_sizes[phase]
                epoch_acc = running_corrects.double() / self.dataset_sizes[phase]

                logger.info(f"{phase.upper()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

                # Deep Copy Best Model
                if phase == 'val' and epoch_acc > best_acc:
                    best_acc = epoch_acc
                    best_model_wts = copy.deepcopy(self.model.state_dict())
                    logger.info(f"New Best Model! Acc: {best_acc:.4f}")

        time_elapsed = time.time() - since
        logger.info(f"Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s")
        logger.info(f"Best Val Acc: {best_acc:.4f}")

        # Load best weights
        self.model.load_state_dict(best_model_wts)
        return self.model

    def visualize(self, num_images=6):
        """
        Inference and Visualization.
        """
        was_training = self.model.training
        self.model.eval()
        images_so_far = 0

        plt.figure(figsize=(10, 10))

        with torch.no_grad():
            for i, (inputs, labels) in enumerate(self.dataloaders['val']):
                inputs = inputs.to(self.device)
                labels = labels.to(self.device)

                outputs = self.model(inputs)
                _, preds = torch.max(outputs, 1)

                for j in range(inputs.size()[0]):
                    images_so_far += 1
                    ax = plt.subplot(num_images // 2, 2, images_so_far)
                    ax.axis('off')

                    # Denormalize
                    img = inputs[j].cpu().numpy().transpose((1, 2, 0))
                    mean = np.array([0.485, 0.456, 0.406])
                    std = np.array([0.229, 0.224, 0.225])
                    img = std * img + mean
                    img = np.clip(img, 0, 1)

                    ax.set_title(f'Predicted: {self.class_names[preds[j]]}')
                    ax.imshow(img)

                    if images_so_far == num_images:
                        self.model.train(mode=was_training)
                        plt.show()
                        return
        self.model.train(mode=was_training)

    def save_model(self, path: str = "model_best.pth"):
        """
        Saves the model weights to disk.
        Crucial for MLOps: We train once, save, and then deploy the file.
        """
        if self.model is None:
            logger.warning("No model to save!")
            return

        # We save the state_dict (weights), not the whole object
        torch.save(self.model.state_dict(), path)
        logger.info(f"Model weights saved to: {path}")

    def load_model(self, path: str = "model_best.pth"):
        """
        Loads weights from disk.
        This is what the API will do when it starts up.
        """
        if not os.path.exists(path):
            raise FileNotFoundError(f"Model file not found: {path}")

        # We must ensure the architecture (ResNet18) is built first
        if self.model is None:
            self.build_model()

        # Then we load the weights into that architecture
        # map_location ensures it loads on CPU even if trained on GPU
        self.model.load_state_dict(torch.load(path, map_location=self.device))
        logger.info(f"Model loaded successfully from: {path}")

    def predict_image(self, image_path: str) -> str:
        """
        Production Inference.
        Takes a single image path and returns the class name.
        """

        # Check file
        if not os.path.exists(image_path):
            return "Error: Image not found"

        # Load and Preprocess (Same as Validation Transform)
        img = Image.open(image_path).convert('RGB')

        preprocess = transforms.Compose([
            transforms.Resize(256),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])

        # Add batch dimension (1, C, H, W) because model expects a batch
        img_tensor = preprocess(img).unsqueeze(0)
        img_tensor = img_tensor.to(self.device)

        # Inference
        self.model.eval() # Vital: Turn off Dropout!
        with torch.no_grad(): # Vital: Don't calculate gradients!
            outputs = self.model(img_tensor)
            _, preds = torch.max(outputs, 1)

        # Decode
        class_name = self.class_names[preds[0]]
        return class_name

In [ ]:
def run_smoke_test(classifier: BeeAntClassifier):
    """
    A quick health check to ensure the system is stable.
    Runs immediately after training or loading.
    """
    logger.info("Running Smoke Test...")

    # Check if model exists in memory
    if classifier.model is None:
        logger.error("Smoke Test Failed: Model not loaded.")
        return False

    # Check Input Processing (using a fake image tensor)
    try:
        # Create a random noise image (1 batch, 3 channels, 224x224)
        dummy_input = torch.randn(1, 3, 224, 224).to(classifier.device)

        # Run inference
        classifier.model.eval()
        with torch.no_grad():
            output = classifier.model(dummy_input)

        # Sanity Check: Output shape must be (1, 2) -> [Prob_Ant, Prob_Bee]
        if output.shape != (1, 2):
            logger.error(f"Sanity Check Failed: Expected output shape (1, 2), got {output.shape}")
            return False

        logger.info("Smoke Test Passed: Inference pipeline is functional.")
        return True

    except Exception as e:
        logger.error(f"Smoke Test Failed: {e}")
        return False

In [ ]:
if __name__ == "__main__":
    # 1. SETUP
    config = ExperimentConfig(EPOCHS=20)
    classifier = BeeAntClassifier(config)

    # 2. TRAIN & SAVE (The "Factory" Phase)
    # This creates the "brain" file
    print("Starting Factory...")
    classifier.prepare_data()
    classifier.build_model()
    classifier.train()
    classifier.save_model("best_model.pth") # <--- CRITICAL STEP

    # 3. VERIFY (The "Quality Control" Phase)
    # We simulate what Docker will do: Start fresh and load the file
    print("Starting Quality Control...")
    production_model = BeeAntClassifier(config)
    production_model.load_model("best_model.pth") # <--- Testing the file works

    # 4. SMOKE TEST
    # Does it predict without crashing?
    if run_smoke_test(production_model):
        print("MODEL IS READY FOR DOCKER.")
    else:
        print("MODEL DEFECTIVE. DO NOT DEPLOY.")

15:03:50 - [INFO] - System initialized on: cuda:0
Starting Factory...
15:03:50 - [INFO] - Dataset found. Skipping download.
15:03:50 - [INFO] - Data Loaded. Classes: ['ants', 'bees']
15:03:50 - [INFO] - Building Model Architecture...
15:03:50 - [INFO] - Model Ready.
15:03:50 - [INFO] - Starting Training for 20 epochs...
15:03:50 - [INFO] - --- Epoch 1/20 ---


15:03:52 - [INFO] - TRAIN Loss: 0.6196 Acc: 0.6967


15:03:54 - [INFO] - VAL Loss: 0.4951 Acc: 0.8105
15:03:54 - [INFO] - New Best Model! Acc: 0.8105
15:03:54 - [INFO] - --- Epoch 2/20 ---


15:03:55 - [INFO] - TRAIN Loss: 0.5042 Acc: 0.7787


15:03:56 - [INFO] - VAL Loss: 0.3815 Acc: 0.8954


...

15:04:41 - [INFO] - VAL Loss: 0.2200 Acc: 0.9346
15:04:41 - [INFO] - --- Epoch 19/20 ---


15:04:43 - [INFO] - TRAIN Loss: 0.2567 Acc: 0.9303


15:04:44 - [INFO] - VAL Loss: 0.2213 Acc: 0.9346
15:04:44 - [INFO] - --- Epoch 20/20 ---


15:04:46 - [INFO] - TRAIN Loss: 0.2368 Acc: 0.9098


15:04:47 - [INFO] - VAL Loss: 0.2189 Acc: 0.9281
15:04:47 - [INFO] - Training complete in 0m 57s
15:04:47 - [INFO] - Best Val Acc: 0.9412
15:04:47 - [INFO] - Model weights saved to: best_model.pth
Starting Quality Control...
15:04:47 - [INFO] - System initialized on: cuda:0
15:04:47 - [INFO] - Building Model Architecture...
15:04:47 - [INFO] - Model Ready.
15:04:47 - [INFO] - Model loaded successfully from: best_model.pth
15:04:47 - [INFO] - Running Smoke Test...
15:04:47 - [INFO] - Smoke Test Passed: Inference pipeline is functional.
MODEL IS READY FOR DOCKER.
